In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql import functions as F
from pyspark.sql.types import DateType, TimestampNTZType,TimestampType
from pyspark.sql.functions import to_date, col

StatementMeta(, 67ffdf92-336b-422f-98ab-a323d1b33038, 88, Finished, Available, Finished, False)

In [ ]:
base_abfss = "abfss://PRUEBA_NS@onelake.dfs.fabric.microsoft.com/SILVER.Lakehouse/Files"


StatementMeta(, c8e43f2d-0bbc-444b-a7d3-ac81f8d7e457, 3, Finished, Available, Finished, False)

In [ ]:
# 1. Configurar las rutas base de origen y destino
# Modifica estas rutas según tu entorno (Fabric, Databricks, Synapse, etc.)
RUTA_LAKEHOUSE_ORIGEN = "abfss://PRUEBA_NS@onelake.dfs.fabric.microsoft.com/RAW.Lakehouse/"
RUTA_LAKEHOUSE_DESTINO ="abfss://PRUEBA_NS@onelake.dfs.fabric.microsoft.com/SILVER.Lakehouse/"

# 2. Definir las tablas y archivos Parquet que deseas copiar
# Formato: (Nombre o ruta relativa en origen, Nombre o ruta en destino)
tablas_a_copiar = [
    ("Tables/dbo/olist_customers_dataset", "Tables/dbo/olist_customers_dataset"),
    ("Tables/dbo/olist_geolocation_dataset", "Tables/dbo/olist_geolocation_dataset"),
    ("Tables/dbo/olist_sellers_dataset", "Tables/dbo/olist_sellers_dataset")
   
]


archivos_parquet_a_copiar =  [
    ("Files/olist/parquet/olist_customers_dataset", "Files/olist_customers_dataset"),
    ("Files/olist/parquet/olist_geolocation_dataset", "Files/olist_geolocation_dataset"),
    ("Files/olist/parquet/olist_sellers_dataset", "Files/olist_sellers_dataset")
   
]

# --- PROCESO 1: Copiar Tablas Delta / Administradas ---
print("--- Iniciando copia de tablas ---")
for origen, destino in tablas_a_copiar:
    ruta_origen = f"{RUTA_LAKEHOUSE_ORIGEN}/{origen}"
    ruta_destino = f"{RUTA_LAKEHOUSE_DESTINO}/{destino}"
    
    print(f"Copiando tabla: {origen} -> {destino}")
    
    # Leer como tabla o formato delta
    df = spark.read.format("delta").load(ruta_origen)

    # 1. Identificar columnas timestamp
    columnas_timestamp = [
        f.name for f in df.schema.fields
        if isinstance(f.dataType, (TimestampType, TimestampNTZType))
    ]

    # 2. Diccionario: nombre nueva columna -> expresión
    transformaciones = {
        f"{c}_date": F.col(c).cast("date") for c in columnas_timestamp
    }

    # 3. Aplicar todas juntas
    df = df.withColumns(transformaciones)
    
    # Guardar en el destino (se usa 'overwrite' para sobrescribir o 'append' para añadir)
    df.write.format("delta").mode("overwrite").save(ruta_destino)

# --- PROCESO 2: Copiar Archivos Parquet puros ---
print("\n--- Iniciando copia de archivos Parquet ---")
for origen, destino in archivos_parquet_a_copiar:
    ruta_origen = f"{RUTA_LAKEHOUSE_ORIGEN}/{origen}"
    ruta_destino = f"{RUTA_LAKEHOUSE_DESTINO}/{destino}"
    
    print(f"Copiando parquet: {origen} -> {destino}")
    
    # Leer los archivos parquet
    df_parquet = spark.read.format("parquet").load(ruta_origen)
    
    # Guardar en el nuevo destino manteniendo el formato parquet
    df_parquet.write.format("parquet").mode("overwrite").save(ruta_destino)

print("\n¡Proceso de copia completado con éxito!")


StatementMeta(, 67ffdf92-336b-422f-98ab-a323d1b33038, 90, Finished, Available, Finished, False)

--- Iniciando copia de tablas ---
Copiando tabla: Tables/dbo/olist_customers_dataset -> Tables/dbo/olist_customers_dataset


Copiando tabla: Tables/dbo/olist_geolocation_dataset -> Tables/dbo/olist_geolocation_dataset


Copiando tabla: Tables/dbo/olist_sellers_dataset -> Tables/dbo/olist_sellers_dataset



--- Iniciando copia de archivos Parquet ---
Copiando parquet: Files/olist/parquet/olist_customers_dataset -> Files/olist_customers_dataset


Copiando parquet: Files/olist/parquet/olist_geolocation_dataset -> Files/olist_geolocation_dataset


Copiando parquet: Files/olist/parquet/olist_sellers_dataset -> Files/olist_sellers_dataset



¡Proceso de copia completado con éxito!


In [ ]:
df = spark.sql("SELECT order_id,sum(payment_value) AS payment_value FROM RAW.dbo.olist_order_payments_dataset GROUP BY order_id")


df.write.mode("overwrite").parquet(f"{base_abfss}/ olist_order_payments_dataset")


# Reemplaza 'Workspace_Name' y 'Lakehouse_Name' por los tuyos
table_name = "olist_order_payments_dataset"
path = (
    f"abfss://PRUEBA_NS@onelake.dfs.fabric.microsoft.com/SILVER.Lakehouse/Tables/dbo/{table_name}"
)

# 1. Identificar columnas timestamp
columnas_timestamp = [
    f.name for f in df.schema.fields
    if isinstance(f.dataType, (TimestampType, TimestampNTZType))
]

# 2. Diccionario: nombre nueva columna -> expresión
transformaciones = {
    f"{c}_date": F.col(c).cast("date") for c in columnas_timestamp
}

# 3. Aplicar todas juntas
df = df.withColumns(transformaciones)

df.write.format("delta").mode("overwrite").save(path)

# Para registrarla también como tabla en el metastore del Lakehouse específico:
df.write.format("delta").mode("overwrite").saveAsTable(f"SILVER.dbo.{table_name}")
    


StatementMeta(, c8e43f2d-0bbc-444b-a7d3-ac81f8d7e457, 4, Finished, Available, Finished, False)

NameError: name 'TimestampType' is not defined

In [ ]:
df = spark.sql("SELECT product_id,product_category_name FROM RAW.dbo.olist_products_dataset GROUP BY product_id,product_category_name")
df.write.mode("overwrite").parquet(f"{base_abfss}/ olist_products_dataset")


# Reemplaza 'Workspace_Name' y 'Lakehouse_Name' por los tuyos
table_name = "olist_products_dataset"
path = (
    f"abfss://PRUEBA_NS@onelake.dfs.fabric.microsoft.com/SILVER.Lakehouse/Tables/dbo/{table_name}"
)

# 1. Identificar columnas timestamp
columnas_timestamp = [
    f.name for f in df.schema.fields
    if isinstance(f.dataType, (TimestampType, TimestampNTZType))
]

# 2. Diccionario: nombre nueva columna -> expresión
transformaciones = {
    f"{c}_date": F.col(c).cast("date") for c in columnas_timestamp
}

# 3. Aplicar todas juntas
df = df.withColumns(transformaciones)

df.write.format("delta").mode("overwrite").save(path)

# Para registrarla también como tabla en el metastore del Lakehouse específico:
df.write.format("delta").mode("overwrite").saveAsTable(f"SILVER.dbo.{table_name}")

StatementMeta(, 67ffdf92-336b-422f-98ab-a323d1b33038, 92, Finished, Available, Finished, False)

In [ ]:
df = spark.sql("SELECT order_id,customer_id,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date FROM RAW.dbo.olist_orders_dataset GROUP BY order_id,customer_id,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date") 
df.write.mode("overwrite").parquet(f"{base_abfss}/ olist_orders_dataset")


# Reemplaza 'Workspace_Name' y 'Lakehouse_Name' por los tuyos
table_name = "olist_orders_dataset"
path = (
    f"abfss://PRUEBA_NS@onelake.dfs.fabric.microsoft.com/SILVER.Lakehouse/Tables/dbo/{table_name}"
)

# 1. Identificar columnas timestamp
columnas_timestamp = [
    f.name for f in df.schema.fields
    if isinstance(f.dataType, (TimestampType, TimestampNTZType))
]

# 2. Diccionario: nombre nueva columna -> expresión
transformaciones = {
    f"{c}_date": F.col(c).cast("date") for c in columnas_timestamp
}

# 3. Aplicar todas juntas
df = df.withColumns(transformaciones)

df.write.format("delta").mode("overwrite").save(path)

# Para registrarla también como tabla en el metastore del Lakehouse específico:
df.write.format("delta").mode("overwrite").saveAsTable(f"SILVER.dbo.{table_name}")

StatementMeta(, 67ffdf92-336b-422f-98ab-a323d1b33038, 93, Finished, Available, Finished, False)

In [ ]:
df = spark.sql("SELECT review_id,order_id,review_score,review_creation_date FROM RAW.dbo.olist_order_reviews_dataset GROUP BY review_id,order_id,review_score,review_creation_date")
df.write.mode("overwrite").parquet(f"{base_abfss}/ olist_reviews_dataset")


# Reemplaza 'Workspace_Name' y 'Lakehouse_Name' por los tuyos
table_name = "olist_reviews_dataset"
path = (
    f"abfss://PRUEBA_NS@onelake.dfs.fabric.microsoft.com/SILVER.Lakehouse/Tables/dbo/{table_name}"
)

# 1. Identificar columnas timestamp
columnas_timestamp = [
    f.name for f in df.schema.fields
    if isinstance(f.dataType, (TimestampType, TimestampNTZType))
]

# 2. Diccionario: nombre nueva columna -> expresión
transformaciones = {
    f"{c}_date": F.col(c).cast("date") for c in columnas_timestamp
}

# 3. Aplicar todas juntas
df = df.withColumns(transformaciones)

df.write.format("delta").mode("overwrite").save(path)

# Para registrarla también como tabla en el metastore del Lakehouse específico:
df.write.format("delta").mode("overwrite").saveAsTable(f"SILVER.dbo.{table_name}")

StatementMeta(, 67ffdf92-336b-422f-98ab-a323d1b33038, 94, Finished, Available, Finished, False)

In [ ]:
df = spark.sql("SELECT order_id,product_id,seller_id FROM RAW.dbo.olist_order_items_dataset GROUP BY  order_id,product_id,seller_id")
df.write.mode("overwrite").parquet(f"{base_abfss}/ olist_order_items_dataset")


# Reemplaza 'Workspace_Name' y 'Lakehouse_Name' por los tuyos
table_name = "olist_order_items_dataset"
path = (
    f"abfss://PRUEBA_NS@onelake.dfs.fabric.microsoft.com/SILVER.Lakehouse/Tables/dbo/{table_name}"
)

# 1. Identificar columnas timestamp
columnas_timestamp = [
    f.name for f in df.schema.fields
    if isinstance(f.dataType, (TimestampType, TimestampNTZType))
]

# 2. Diccionario: nombre nueva columna -> expresión
transformaciones = {
    f"{c}_date": F.col(c).cast("date") for c in columnas_timestamp
}

# 3. Aplicar todas juntas
df = df.withColumns(transformaciones)

df.write.format("delta").mode("overwrite").save(path)

# Para registrarla también como tabla en el metastore del Lakehouse específico:
df.write.format("delta").mode("overwrite").saveAsTable(f"SILVER.dbo.{table_name}")

StatementMeta(, 67ffdf92-336b-422f-98ab-a323d1b33038, 95, Finished, Available, Finished, False)

In [ ]:
# 1. Generar la secuencia de fechas y desanidarla con explode
calendar_df = spark.sql(
    "SELECT EXPLODE(SEQUENCE(DATE '2016-01-01', DATE '2020-12-31', INTERVAL 1 DAY)) AS full_date"
)

# 2. Agregar columnas descriptivas de calendario
dim_calendar = calendar_df.select(
    F.date_format("full_date", "yyyy-MM-dd").cast("int").alias("date_key"),
    F.col("full_date"),
    F.year("full_date").alias("year"),
    F.quarter("full_date").alias("quarter"),
    F.month("full_date").alias("month"),
    F.date_format("full_date", "MMMM").alias("month_name"),
    F.dayofmonth("full_date").alias("day_of_month"),
    F.weekofyear("full_date").alias("week_of_year"),
    F.dayofweek("full_date").alias("day_of_week"),
    F.date_format("full_date", "EEEE").alias("day_name"),
)

# 3. Mostrar o guardar como tabla (ej. Delta Lake)
dim_calendar.write.format("delta").mode("overwrite").saveAsTable("SILVER.dbo.dim_calendar")

StatementMeta(, 67ffdf92-336b-422f-98ab-a323d1b33038, 96, Finished, Available, Finished, False)

In [ ]:
df = spark.sql("SELECT geolocation_city FROM RAW.dbo.olist_geolocation_dataset GROUP BY geolocation_city")

df.write.mode("overwrite").parquet(f"{base_abfss}/ dim_cities")


# Reemplaza 'Workspace_Name' y 'Lakehouse_Name' por los tuyos
table_name = "dim_cities"
path = (
    f"abfss://PRUEBA_NS@onelake.dfs.fabric.microsoft.com/SILVER.Lakehouse/Tables/dbo/{table_name}"
)

# 1. Identificar columnas timestamp
columnas_timestamp = [
    f.name for f in df.schema.fields
    if isinstance(f.dataType, (TimestampType, TimestampNTZType))
]

# 2. Diccionario: nombre nueva columna -> expresión
transformaciones = {
    f"{c}_date": F.col(c).cast("date") for c in columnas_timestamp
}

# 3. Aplicar todas juntas
df = df.withColumns(transformaciones)

df.write.format("delta").mode("overwrite").save(path)

# Para registrarla también como tabla en el metastore del Lakehouse específico:
df.write.format("delta").mode("overwrite").saveAsTable(f"SILVER.dbo.{table_name}")

StatementMeta(, 67ffdf92-336b-422f-98ab-a323d1b33038, 97, Finished, Available, Finished, False)